# 01 — Metabolite Vector Construction

One row per (RSU, food). Columns are metabolite fields with numeric ranges.
Values are midpoints of reported ranges. Null = unmeasured.

**Output:** `data/metabolites/rsu_metabolite_matrix.csv` and `rsu_distance_matrix.csv`

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from pathlib import Path
from rsu_loader import load_all_rsus, parse_numeric_ranges

In [2]:
rsus = load_all_rsus()
ranges = parse_numeric_ranges(rsus)

# Flatten: one row per (RSU, food)
rows = []
for region_id, foods in ranges.items():
    for food_name, fields in foods.items():
        row = {'region_id': region_id, 'food_name': food_name}
        for field, (lo, hi) in fields.items():
            row[field] = (lo + hi) / 2
        rows.append(row)

mid_df = pd.DataFrame(rows).set_index(['region_id', 'food_name'])
print(f"Matrix shape: {mid_df.shape}  ({mid_df.shape[0]} food rows x {mid_df.shape[1]} metabolite dimensions)")
mid_df

Matrix shape: (268, 34)  (268 food rows x 34 metabolite dimensions)


primary_metabolites.starch_content  \
region_id food_name                                                                                
RSU-01    rye bread                                                                       77.162   
          Atlantic salmon                                                                    NaN   
          pike                                                                               NaN   
          cloudberries                                                                     8.600   
          fermented Baltic herring                                                         9.640   
...                                                                                          ...   
RSU-61    green tea / raw pu-erh (Camellia sinensis var. ...                                 NaN   
RSU-62    Longjing green tea (pan-fired)                                                     NaN   
RSU-63    green tea (Camellia sinensis)                                                      NaN   
RSU-64    Kenya highland green tea (Camellia sinensis)                                       NaN   
RSU-65    Kenya highland green tea (Camellia sinensis)                                       NaN   

                                                              primary_metabolites.lipid_content  \
region_id food_name                                                                               
RSU-01    rye bread                                                                       1.908   
          Atlantic salmon                                                                13.110   
          pike                                                                            0.690   
          cloudberries                                                                    0.800   
          fermented Baltic herring                                                       18.000   
...                                                                                         ...   
RSU-61    green tea / raw pu-erh (Camellia sinensis var. ...                                NaN   
RSU-62    Longjing green tea (pan-fired)                                                    NaN   
RSU-63    green tea (Camellia sinensis)                                                     NaN   
RSU-64    Kenya highland green tea (Camellia sinensis)                                      NaN   
RSU-65    Kenya highland green tea (Camellia sinensis)                                      NaN   

                                                              primary_metabolites.protein_content  \
region_id food_name                                                                                 
RSU-01    rye bread                                                                         8.395   
          Atlantic salmon                                                                  20.319   
          pike                                                                             19.260   
          cloudberries                                                                      2.400   
          fermented Baltic herring                                                         14.190   
...                                                                                           ...   
RSU-61    green tea / raw pu-erh (Camellia sinensis var. ...                                  NaN   
RSU-62    Longjing green tea (pan-fired)                                                      NaN   
RSU-63    green tea (Camellia sinensis)                                                       NaN   
RSU-64    Kenya highland green tea (Camellia sinensis)                                        NaN   
RSU-65    Kenya highland green tea (Camellia sinensis)                                        NaN   

                                                              organic_acids.lactic_acid  \
region_id food_name                                                 

## Shared dimensions

Dimensions observed in >= 2 foods across the dataset.

In [3]:
observed_counts = mid_df.notna().sum()
shared_dims = observed_counts[observed_counts >= 2].index.tolist()

print(f"Shared dimensions (observed in >= 2 foods): {len(shared_dims)}")
for d in shared_dims:
    print(f"  {d}  -- observed in {int(observed_counts[d])} foods")

Shared dimensions (observed in >= 2 foods): 33
  primary_metabolites.starch_content  -- observed in 91 foods
  primary_metabolites.lipid_content  -- observed in 170 foods
  primary_metabolites.protein_content  -- observed in 157 foods
  organic_acids.lactic_acid  -- observed in 46 foods
  organic_acids.citric_acid  -- observed in 28 foods
  organic_acids.malic_acid  -- observed in 31 foods
  primary_metabolites.glucose_concentration  -- observed in 17 foods
  key_flavor_bioactives.tannin_content  -- observed in 18 foods
  organic_acids.acetic_acid  -- observed in 30 foods
  umami_compounds.glutamate  -- observed in 86 foods
  terpenes.pinene  -- observed in 4 foods
  terpenes.linalool  -- observed in 22 foods
  primary_metabolites.fructose_concentration  -- observed in 13 foods
  primary_metabolites.ascorbic_acid  -- observed in 26 foods
  terpenes.limonene  -- observed in 9 foods
  key_flavor_bioactives.capsaicinoids  -- observed in 10 foods
  terpenes.myrcene  -- observed in 6 foods


In [4]:
shared_matrix = mid_df[shared_dims]

out_path = Path('../data/metabolites')
import os
out_path.mkdir(exist_ok=True)
shared_matrix.to_csv(os.path.join(out_path, 'rsu_metabolite_matrix.csv'))
print(f"Saved {shared_matrix.shape[0]} rows x {shared_matrix.shape[1]} dims")

Saved 268 rows x 33 dims


## Pairwise distance matrix

Euclidean distance across shared observed dimensions.
Column-mean fill used only for distance computation — not imputation for the model.
Index is flattened to `RSU-XX|food name` strings so the matrix saves and reloads cleanly.

In [5]:
from scipy.spatial.distance import pdist, squareform

valid = shared_matrix.dropna(how='all')
print(f"Food rows with at least one numeric field: {len(valid)}")

valid_filled = valid.apply(lambda col: col.fillna(col.mean()))

# Flatten MultiIndex to plain strings: 'RSU-01|rye bread'
flat_index = [f"{r}|{f}" for r, f in valid.index]

dist_matrix = pd.DataFrame(
    squareform(pdist(valid_filled.values, metric='euclidean')),
    index=flat_index,
    columns=flat_index
)
dist_matrix.to_csv(out_path / 'rsu_distance_matrix.csv')
print(f"Distance matrix saved: {dist_matrix.shape}")
dist_matrix

Food rows with at least one numeric field: 268
Distance matrix saved: (268, 268)


,RSU-01|rye bread,RSU-01|Atlantic salmon,RSU-01|pike,RSU-01|cloudberries,RSU-01|fermented Baltic herring,RSU-02|bannock,RSU-02|moose,RSU-02|walleye,RSU-02|wild blueberries,RSU-02|smoked whitefish,...,RSU-56|green tea (Camellia sinensis),RSU-57|green tea (Camellia sinensis),RSU-58|green tea (Camellia sinensis),RSU-59|green tea (Camellia sinensis),RSU-60|green tea (Camellia sinensis),RSU-61|green tea / raw pu-erh (Camellia sinensis var. assamica),RSU-62|Longjing green tea (pan-fired),RSU-63|green tea (Camellia sinensis),RSU-64|Kenya highland green tea (Camellia sinensis),RSU-65|Kenya highland green tea (Camellia sinensis)
RSU-01|rye bread,0.000000,56.821174,55.502351,81.717375,82.434375,62.166778,56.160729,81.623746,337.704107,56.454280,...,1073.396862,1068.990543,1069.963737,1072.757663,1072.884149,1071.414462,1073.663549,1070.686536,1071.447683,1068.953989
RSU-01|Atlantic salmon,56.821174,0.000000,12.465066,42.575082,36.473115,24.487586,12.518272,42.865288,331.077337,12.563636,...,1071.838258,1067.425505,1068.400125,1071.198130,1071.324800,1069.852971,1072.105334,1069.123982,1069.886240,1067.388898
RSU-01|pike,55.502351,12.465066,0.000000,40.302390,39.926838,20.415580,2.980419,40.437602,330.768776,4.146951,...,1072.054729,1067.642871,1068.617293,1071.414730,1071.541375,1070.069844,1072.321751,1069.341002,1070.103106,1067.606271
RSU-01|cloudberries,81.717375,42.575082,40.302390,0.000000,20.974900,25.688730,41.637004,0.933584,328.882750,42.202225,...,1072.708905,1068.299749,1069.273572,1072.069296,1072.195864,1070.725232,1072.975764,1069.996837,1070.758474,1068.263171
RSU-01|fermented Baltic herring,82.434375,36.473115,39.926838,20.974900,0.000000,32.362449,40.392097,21.775809,329.618462,40.558851,...,1072.356476,1067.945865,1068.920010,1071.716657,1071.843266,1070.372150,1072.623423,1069.643515,1070.405403,1067.909275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RSU-61|green tea / raw pu-erh (Camellia sinensis var. assamica),1071.414462,1069.852971,1070.069844,1070.725232,1070.372150,1070.159861,1070.090611,1070.737930,1119.928434,1070.096903,...,2.392175,2.623928,1.453444,93.645707,89.639958,0.000000,2.275983,1.110732,0.474342,2.494995
RSU-62|Longjing green tea (pan-fired),1073.663549,1072.105334,1072.321751,1072.975764,1072.623423,1072.411579,1072.342475,1072.988435,1122.080285,1072.348754,...,1.551644,4.810208,3.718414,93.740458,89.726680,2.275983,0.000000,3.152876,2.285629,4.745956
RSU-63|green tea (Camellia sinensis),1070.686536,1069.123982,1069.341002,1069.996837,1069.643515,1069.431081,1069.361784,1070.009544,1119.232061,1069.368080,...,2.760657,1.720966,1.092119,93.627852,89.618822,1.110732,3.152876,0.000000,0.913906,1.805609
RSU-64|Kenya highland green tea (Camellia sinensis),1071.447683,1069.886240,1070.103106,1070.758474,1070.405403,1070.193121,1070.123873,1070.771172,1119.960216,1070.130165,...,2.191461,2.524876,1.550000,93.646801,89.636583,0.474342,2.285629,0.913906,0.000000,2.500000
